In [0]:
from pyspark.sql import functions as F

RAW_PATH = "/Volumes/workspace/linguacopilot/raw_events/events/"
CHECKPOINT = "/Volumes/workspace/linguacopilot/checkpoints/bronze_events"
BRONZE_TABLE = "workspace.linguacopilot.bronze_events"

(
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT}/schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(RAW_PATH)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_bronze_loaded_at", F.current_timestamp())
    .writeStream
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(BRONZE_TABLE)
)

In [0]:
display(spark.table("workspace.linguacopilot.bronze_events"))